# Notebook 06 — Modèles Dynamiques & ARDL Parcimonieux (2000–2025)

## Objectifs Économétriques
Tester si les lumières nocturnes contiennent de l'information sur l'activité économique **de l'année suivante ($t-1$)** :

1. **Modèle 5 (Croissance avec NTL Retardé $t-1$)** :
   $$\Delta \ln(GDP_t) = \beta_0 + \beta_1 \Delta \ln(NTL_{t-1}) + \gamma D_{2017} + \varepsilon_t$$
2. **Modèle 6 (Dynamique Croisée : $NTL_t$ + $NTL_{t-1}$)** :
   $$\Delta \ln(GDP_t) = \beta_0 + \beta_1 \Delta \ln(NTL_t) + \beta_2 \Delta \ln(NTL_{t-1}) + \gamma D_{2017} + \varepsilon_t$$
3. **Modèle 7 (Autorégressif ARDL(1,1) en Croissance)** :
   $$\Delta \ln(GDP_t) = \beta_0 + \rho \Delta \ln(GDP_{t-1}) + \beta_1 \Delta \ln(NTL_t) + \beta_2 \Delta \ln(NTL_{t-1}) + \gamma D_{2017} + \varepsilon_t$$

In [1]:
# ==============================================================================
# 1. ESTIMATION DES MODÈLES DYNAMIQUES ET RETARDÉS (HAC / NEWEY-WEST)
# ==============================================================================
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.5f' % x)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
tables_dir = project_root / "outputs" / "tables"
figures_dir = project_root / "outputs" / "figures"

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Chargement du jeu de données audité
df = pd.read_csv(project_root / "data" / "intermediate" / "ntl_processed" / "niger_ntl_audited_2000_2025.csv")

# Transformations Logarithmiques et Différences
df['ln_gdp'] = np.log(df['gdp_real'])
df['ln_ntl'] = np.log(df['ntl_sum_adj'])

df['d_ln_gdp'] = df['ln_gdp'].diff()
df['d_ln_ntl'] = df['ln_ntl'].diff()

# Création des retards temporels (Lags t-1)
df['d_ln_gdp_lag1'] = df['d_ln_gdp'].shift(1)
df['d_ln_ntl_lag1'] = df['d_ln_ntl'].shift(1)

# Nettoyage de l'échantillon pour régressions dynamiques (suppression des NaNs dus aux lags)
df_dyn = df.dropna(subset=['d_ln_gdp', 'd_ln_ntl', 'd_ln_gdp_lag1', 'd_ln_ntl_lag1']).copy()

# ------------------------------------------------------------------------------
# ESTIMATIONS OLS HAC ROBUSTES
# ------------------------------------------------------------------------------

# Modèle 5 : Croissance GDP vs Croissance NTL retardée (t-1) + Control 2017
m5 = smf.ols('d_ln_gdp ~ d_ln_ntl_lag1 + dummy_2017', data=df_dyn).fit(cov_type='HAC', cov_kwds={'maxlags': 2})

# Modèle 6 : Effet Contemporain (t) + Effet Retardé (t-1)
m6 = smf.ols('d_ln_gdp ~ d_ln_ntl + d_ln_ntl_lag1 + dummy_2017', data=df_dyn).fit(cov_type='HAC', cov_kwds={'maxlags': 2})

# Modèle 7 : ARDL(1,1) Parcimonieux (gdp_lag1 + ntl + ntl_lag1 + dummy_2017)
m7 = smf.ols('d_ln_gdp ~ d_ln_gdp_lag1 + d_ln_ntl + d_ln_ntl_lag1 + dummy_2017', data=df_dyn).fit(cov_type='HAC', cov_kwds={'maxlags': 2})

print(f"[✓] 3 Modèles dynamiques estimés avec succès sur N = {len(df_dyn)} observations (2002-2025).")

[✓] 3 Modèles dynamiques estimés avec succès sur N = 24 observations (2002-2025).


## Section 2 : Évaluation du Pouvoir Explicatif Temporel ($NTL_{t-1}$)

Nous analysons si les variations de luminosité de l'année précédente ($t-1$) apportent une information significative pour expliquer la croissance économique contemporaine ($t$), validant l'utilité des données satellitaires comme indicateur d'activité à court terme.

In [2]:
# ==============================================================================
# 2. TABLEAU COMPARATIF DES ESTIMATIONS DYNAMIQUES (M5, M6, M7)
# ==============================================================================

def extract_dynamic_summary(model, model_name):
    """Extrait les statistiques des modèles dynamiques."""
    p_values = model.pvalues
    params = model.params
    std_errs = model.bse
    
    # NTL Lag 1
    b_lag = params.get('d_ln_ntl_lag1', np.nan)
    p_lag = p_values.get('d_ln_ntl_lag1', np.nan)
    
    # NTL Contemporain
    b_cont = params.get('d_ln_ntl', np.nan)
    p_cont = p_values.get('d_ln_ntl', np.nan)
    
    # AR(1) GDP Lag
    b_ar = params.get('d_ln_gdp_lag1', np.nan)
    p_ar = p_values.get('d_ln_gdp_lag1', np.nan)
    
    return {
        'Modèle': model_name,
        'Coef Δln(NTL)_t': b_cont,
        'p-val (t)': p_cont,
        'Coef Δln(NTL)_t-1': b_lag,
        'p-val (t-1)': p_lag,
        'Coef Δln(GDP)_t-1': b_ar,
        'p-val AR(1)': p_ar,
        'R²': model.rsquared,
        'R² Ajusté': model.rsquared_adj,
        'F-stat (p-value)': model.f_pvalue,
        'Observations': int(model.nobs)
    }

dyn_summary = [
    extract_dynamic_summary(m5, 'Modèle 5 (NTL Retardé t-1)'),
    extract_dynamic_summary(m6, 'Modèle 6 (NTL Contemporain + Retardé)'),
    extract_dynamic_summary(m7, 'Modèle 7 (ARDL 1,1 Parcimonieux)')
]

df_dyn_summary = pd.DataFrame(dyn_summary)

print("=======================================================================================")
print(" SUMMARY DES ESTIMATIONS DYNAMIQUES (HAC / NEWEY-WEST)")
print("=======================================================================================")
print(df_dyn_summary[['Modèle', 'Coef Δln(NTL)_t', 'p-val (t)', 'Coef Δln(NTL)_t-1', 'p-val (t-1)', 'R²', 'R² Ajusté']])

# Exportation du tableau récapitulatif
tables_dir = project_root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
out_dyn_path = tables_dir / "06_dynamic_models_summary.csv"
df_dyn_summary.to_csv(out_dyn_path, index=False)

print(f"\n[✓] Tableau récapitulatif dynamique exporté vers : {out_dyn_path}")

 SUMMARY DES ESTIMATIONS DYNAMIQUES (HAC / NEWEY-WEST)
                                  Modèle  Coef Δln(NTL)_t  p-val (t)  \
0             Modèle 5 (NTL Retardé t-1)              NaN        NaN   
1  Modèle 6 (NTL Contemporain + Retardé)          0.00576    0.52994   
2       Modèle 7 (ARDL 1,1 Parcimonieux)          0.00891    0.18904   

   Coef Δln(NTL)_t-1  p-val (t-1)      R²  R² Ajusté  
0            0.00339      0.64938 0.00327   -0.09166  
1            0.00476      0.53394 0.00735   -0.14155  
2            0.00534      0.47043 0.21422    0.04879  

[✓] Tableau récapitulatif dynamique exporté vers : c:\Niger & Big Data\NTL et PIB Niger\niger_ntl_gdp\outputs\tables\06_dynamic_models_summary.csv
